# 01 - Data Cleaning, Quality Audits & Normalization Pipeline
### Project: E-Commerce Customer, Sales & Business Intelligence Analytics
**Author:** Senior Data Analyst & BI Developer  
**Dataset:** Brazilian E-Commerce Public Dataset by Olist  

---

## 1. Objectives & Pipeline Overview
The goal of this notebook is to execute an audit and cleaning process across all 9 relational entities of the Brazilian Olist E-Commerce dataset:
1. **Schema & Data Types**: Parse date fields to `datetime64[ns]` and validate numeric keys.
2. **Missing Value Imputation**: Translate Portuguese category names into English and handle null dimensions.
3. **Integrity & Deduplication**: Verify primary keys, eliminate duplicate transactions, and resolve duplicate geolocation entries.
4. **Chronological Validity**: Remove chronological anomalies (e.g. delivered prior to purchase).
5. **Data Export**: Save cleansed datasets into `data/processed/` for SQL ingestion and Power BI modeling.


In [ ]:
import os
import pandas as pd
import numpy as np

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

RAW_DATA_DIR = '../data/raw'
PROCESSED_DATA_DIR = '../data/processed'
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
print("Data directories initialized successfully.")


## 2. Ingestion and Inspection of Raw Datasets

In [ ]:
df_orders_raw = pd.read_csv(os.path.join(RAW_DATA_DIR, 'olist_orders_dataset.csv'))
df_items_raw = pd.read_csv(os.path.join(RAW_DATA_DIR, 'olist_order_items_dataset.csv'))
df_products_raw = pd.read_csv(os.path.join(RAW_DATA_DIR, 'olist_products_dataset.csv'))
df_trans_raw = pd.read_csv(os.path.join(RAW_DATA_DIR, 'product_category_name_translation.csv'))
df_customers_raw = pd.read_csv(os.path.join(RAW_DATA_DIR, 'olist_customers_dataset.csv'))
df_sellers_raw = pd.read_csv(os.path.join(RAW_DATA_DIR, 'olist_sellers_dataset.csv'))
df_payments_raw = pd.read_csv(os.path.join(RAW_DATA_DIR, 'olist_order_payments_dataset.csv'))
df_reviews_raw = pd.read_csv(os.path.join(RAW_DATA_DIR, 'olist_order_reviews_dataset.csv'))
df_geo_raw = pd.read_csv(os.path.join(RAW_DATA_DIR, 'olist_geolocation_dataset.csv'))

raw_summary = pd.DataFrame({
    'Dataset': ['Orders', 'Order Items', 'Products', 'Customers', 'Sellers', 'Payments', 'Reviews', 'Geolocation', 'Category Translations'],
    'Row Count': [len(df_orders_raw), len(df_items_raw), len(df_products_raw), len(df_customers_raw), len(df_sellers_raw), len(df_payments_raw), len(df_reviews_raw), len(df_geo_raw), len(df_trans_raw)],
    'Column Count': [df_orders_raw.shape[1], df_items_raw.shape[1], df_products_raw.shape[1], df_customers_raw.shape[1], df_sellers_raw.shape[1], df_payments_raw.shape[1], df_reviews_raw.shape[1], df_geo_raw.shape[1], df_trans_raw.shape[1]]
})
raw_summary


## 3. Cleansing & Transforming Orders Dataset

In [ ]:
df_orders = df_orders_raw.copy().drop_duplicates(subset=['order_id'])

# Datetime conversions
date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    df_orders[col] = pd.to_datetime(df_orders[col], errors='coerce')

df_orders['order_status'] = df_orders['order_status'].str.strip().str.lower()

# Chronological validation
invalid_mask = (
    (df_orders['order_status'] == 'delivered') &
    (df_orders['order_delivered_customer_date'].notna()) &
    (df_orders['order_delivered_customer_date'] < df_orders['order_purchase_timestamp'])
)
df_orders = df_orders[~invalid_mask]
print(f"Cleaned orders shape: {df_orders.shape}")
df_orders.head(3)


## 4. Product Category Translation & Dimension Imputation

In [ ]:
df_products = df_products_raw.copy().drop_duplicates(subset=['product_id'])
df_products = df_products.merge(df_trans_raw, on='product_category_name', how='left')

df_products['product_category_name_english'] = (
    df_products['product_category_name_english']
    .fillna('other_uncategorized')
    .str.replace('_', ' ')
    .str.title()
)
df_products['product_category_name'] = df_products['product_category_name'].fillna('outro')

# Impute dimensions with medians
dim_cols = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty', 
            'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in dim_cols:
    df_products[col] = df_products[col].fillna(df_products[col].median())

print(f"Cleaned products shape: {df_products.shape}")
df_products.head(3)


## 5. Order Items, Payments & Reviews Validation

In [ ]:
# Items
df_items = df_items_raw.copy().drop_duplicates(subset=['order_id', 'order_item_id'])
df_items['shipping_limit_date'] = pd.to_datetime(df_items['shipping_limit_date'], errors='coerce')
df_items = df_items[(df_items['price'] > 0) & (df_items['freight_value'] >= 0)]

# Payments
df_payments = df_payments_raw.copy().drop_duplicates()
df_payments['payment_type'] = df_payments['payment_type'].str.strip().str.lower().str.replace('_', ' ').str.title()
df_payments = df_payments[df_payments['payment_type'] != 'Not Defined']
df_payments = df_payments[df_payments['payment_value'] >= 0]

# Reviews
df_reviews = df_reviews_raw.copy()
df_reviews['review_creation_date'] = pd.to_datetime(df_reviews['review_creation_date'], errors='coerce')
df_reviews['review_answer_timestamp'] = pd.to_datetime(df_reviews['review_answer_timestamp'], errors='coerce')
df_reviews = df_reviews.sort_values('review_answer_timestamp', ascending=False).drop_duplicates(subset=['order_id', 'review_id'])
df_reviews = df_reviews[df_reviews['review_score'].between(1, 5)]
df_reviews['review_comment_title'] = df_reviews['review_comment_title'].fillna('')
df_reviews['review_comment_message'] = df_reviews['review_comment_message'].fillna('')

print("Items:", df_items.shape, "| Payments:", df_payments.shape, "| Reviews:", df_reviews.shape)


## 6. Customers, Sellers & Geolocation Standardization

In [ ]:
df_customers = df_customers_raw.copy().drop_duplicates(subset=['customer_id'])
df_customers['customer_city'] = df_customers['customer_city'].str.strip().str.title()
df_customers['customer_state'] = df_customers['customer_state'].str.strip().str.upper()

df_sellers = df_sellers_raw.copy().drop_duplicates(subset=['seller_id'])
df_sellers['seller_city'] = df_sellers['seller_city'].str.strip().str.title()
df_sellers['seller_state'] = df_sellers['seller_state'].str.strip().str.upper()

# Geolocation aggregation
valid_coords = (
    (df_geo_raw['geolocation_lat'] >= -35.0) & (df_geo_raw['geolocation_lat'] <= 6.0) &
    (df_geo_raw['geolocation_lng'] >= -75.0) & (df_geo_raw['geolocation_lng'] <= -30.0)
)
df_geo = df_geo_raw[valid_coords].groupby('geolocation_zip_code_prefix').agg(
    geolocation_lat=('geolocation_lat', 'mean'),
    geolocation_lng=('geolocation_lng', 'mean'),
    geolocation_city=('geolocation_city', 'first'),
    geolocation_state=('geolocation_state', 'first')
).reset_index()

print("Customers:", df_customers.shape, "| Sellers:", df_sellers.shape, "| Geolocation:", df_geo.shape)


## 7. Exporting Cleaned Processed Layer

In [ ]:
df_orders.to_csv(os.path.join(PROCESSED_DATA_DIR, 'fact_orders_clean.csv'), index=False)
df_items.to_csv(os.path.join(PROCESSED_DATA_DIR, 'fact_order_items_clean.csv'), index=False)
df_products.to_csv(os.path.join(PROCESSED_DATA_DIR, 'dim_products_clean.csv'), index=False)
df_customers.to_csv(os.path.join(PROCESSED_DATA_DIR, 'dim_customers_clean.csv'), index=False)
df_sellers.to_csv(os.path.join(PROCESSED_DATA_DIR, 'dim_sellers_clean.csv'), index=False)
df_payments.to_csv(os.path.join(PROCESSED_DATA_DIR, 'fact_order_payments_clean.csv'), index=False)
df_reviews.to_csv(os.path.join(PROCESSED_DATA_DIR, 'fact_order_reviews_clean.csv'), index=False)
df_geo.to_csv(os.path.join(PROCESSED_DATA_DIR, 'dim_geolocation_clean.csv'), index=False)
print("Data cleaning completed successfully. 8 normalized files saved.")
